# Notebook07 — D200 TAIL expanded development50

**Fresh Colab T4 → Run All.300 NEW episodes. No training.**

Archive the previous completed development10 evaluator as:
`07_Closed_Loop_Development_Pilot_D200_TAIL_SEEDS42_43_44_DEV10_COMPLETED_ARCHIVE_2026-10-12.ipynb`.
Keep this file named `07_Closed_Loop_Development_Pilot.ipynb`.

The six SMALL RGB/Event models from training seeds42/43/44 are fixed at their exact
FP32 EMA best/final4500 generations. The preceding ten-reset pilot completed:
RGB sustained10/10 and Events8/10 in each training seed. Those60 episodes remain
historical and are excluded from this new evaluation.

## Frozen scientific protocol

-50 new reset seeds2100–2149, paired policy sampling seeds3100–3149 for ALL six models.
  Final evaluation seeds1000–1049 and previous development seeds are untouched.
- Fixed TRAIN `demo_10` scene geometry; fresh seeded reset. This does not expand scene
  geometry or initialize from a recorded validation/test demonstration state.
-300-step /15 simulated-second horizon at20Hz; execute indices0:8 of the16-action plan.
- Primary: environment Lift predicate for10 consecutive steps (0.5 simulated seconds).
  Any-success is secondary. COMPLETED is execution status; Lift tests cube height,
  not a verified gripper attachment. Late Lift at the horizon is not retroactively rescored.
- Same100-step FP32 EMA epsilon DDPM with explicit TRAIN-range x0 projection in z-score
  coordinates. No sampler, checkpoint, architecture, vision encoder or training change.
- Both modalities retain the exercised three-native-draw source, persistent primary/
  shadow CPU V2E, strict source/physics/render gates and dual Event tensor parity.
- Real RGB reset duplication / zero Events startup, then causal20Hz observations.

## Run All and resume

The notebook validates all six training records, pinned scientific reference, all200
paired source exports and selected weights. Each seed uses disposable preflight and
fresh isolated worker processes. Each completed seed42 RGB reset establishes a new
checksummed reset-bank record; every other modality/training seed must match it
before rollout. No past episode is copied into the new score.

Episode evidence is saved after each completion. Each training seed runs100 episodes:
50 RGB +50 Events. Stop between episodes when practical; after a runtime interruption,
Run All reuses only verified completed episodes and restarts unfinished episodes.
Paired comparison and graphs refresh every10 completions; final publication rechecks
all artifacts. Expensive ZIP rebuilding occurs at seed completion and finalization,
not after every episode. Model/source gates remain mandatory for new scoring.

## Resource control

`AUTO_DISCONNECT_ON_COMPLETE=True` is the default. After all300 episodes, summaries,
videos and both bundles are saved and verified, the final cell flushes/unmounts Drive
and requests Colab runtime disconnection. No automatic disconnection follows a partial,
failed or manually interrupted run. If the Colab service API is unavailable or fails,
the notebook prints a manual-disconnection instruction; do not assume it disconnected.
Set the flag False before Run All if you want the runtime to remain open.

This is expanded DEVELOPMENT evaluation, not the final locked test or robustness study.
Results remain conditional on this reset distribution, three trained models, one scene
and synthetic Events.300 trials are not300 independent model replications.


## 1. Configuration and Drive

Choose T4. Keep the fixed development seeds and scientific settings.

In [ ]:
PROJECT_DIRECTORY='/content/drive/MyDrive/DiffusionPolicyEvents'
OPTIONS={'scene_demo': 'demo_10', 'environment_seeds': [2100, 2101, 2102, 2103, 2104, 2105, 2106, 2107, 2108, 2109, 2110, 2111, 2112, 2113, 2114, 2115, 2116, 2117, 2118, 2119, 2120, 2121, 2122, 2123, 2124, 2125, 2126, 2127, 2128, 2129, 2130, 2131, 2132, 2133, 2134, 2135, 2136, 2137, 2138, 2139, 2140, 2141, 2142, 2143, 2144, 2145, 2146, 2147, 2148, 2149], 'policy_seeds': [3100, 3101, 3102, 3103, 3104, 3105, 3106, 3107, 3108, 3109, 3110, 3111, 3112, 3113, 3114, 3115, 3116, 3117, 3118, 3119, 3120, 3121, 3122, 3123, 3124, 3125, 3126, 3127, 3128, 3129, 3130, 3131, 3132, 3133, 3134, 3135, 3136, 3137, 3138, 3139, 3140, 3141, 3142, 3143, 3144, 3145, 3146, 3147, 3148, 3149], 'max_control_steps': 300, 'execution_horizon': 8, 'success_hold_steps': 10, 'video_stride': 2, 'source_policy': 'strict', 'sampler': 'train_range_x0', 'nvidia_library': '/usr/lib64-nvidia/libEGL_nvidia.so.0', 'max_frames_per_camera': 1501, 'episode_timeout_seconds': 900}
AUTO_DISCONNECT_ON_COMPLETE=True
import ast,base64,hashlib,json,math,os,shutil,sys,tempfile,time,traceback,uuid,zipfile,zlib
from pathlib import Path
from datetime import datetime,timezone
from google.colab import drive
drive.mount('/content/drive')
PROJECT_ROOT=Path(PROJECT_DIRECTORY)
assert (PROJECT_ROOT/'data/raw/lift_ph_raw.hdf5').is_file(), 'Verified project Drive directory is required.'
SESSION=PROJECT_ROOT/'results/closed_loop_development/notebook07_development50_sessions'/(datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'_'+uuid.uuid4().hex[:8])
SESSION.mkdir(parents=True,exist_ok=False)
print('Expanded development50; six fixed models; no training. Session:',SESSION)


## 2. Hashed evaluator and exact six-model catalog

Numerical simulation, source, guard and policy workers are preserved.

In [ ]:
BASE_MODULES=json.loads(zlib.decompress(base64.b64decode('...')).decode())
SOURCE_CATALOG=json.loads(zlib.decompress(base64.b64decode('eNqtlcFu5DYMhl9lMeeiJSmSEvsyBiWKSdBFGmSyQYFF3r30LgrkMIcCsxdbFg3618ef9PcL0+XPL98vrw/z5/3b8/EUtbwEARy1fbz40+uO4+Hl2/HmT1+PfPpnB0tF3/G47lrTkUyEOYbrsMtvXy7r7+d8ejge/fp45vocpiBvibCXNEjAVUvx2F0UMVTIO5K5DO3e+5iqmmfKua9vPyRe3/ZLLU4Ftf26v/rb0/s+P3O+8scZPgB+CATshh3ZmjAjNe6gB2SPFei/r79e3s7M10cn0TMB2HJU3BucGiJrumd3iQWLRylrM2eU7IGThkaIQK4K+gJybJePj8q33/fz2/UWzp+R/0W0j6190hR2vEH0cxh01XFTbZv67N6Gs0HE0m5jiuogFMnYSDykFYgx7iQ6BAikKwgbHsq7N9d9g+iiNtrOpgXTZ4bt0WmjxgxEEC+Szn1jUcXSP6r4Ikq41qaOxkX0RMrtfpe2Y0gWp1XQYN5g+jksq0Sgypo2Z3lwUuRe0MMdUnO1hanunUPQMgdlm/cx5SIJnapadWw4MLY5UdxiGiO8FTOk3lufVv2D1UGr/LgouilAuaBKLgrbcTRaE4ZIt5xu8stc2o5V9UTVyj/iBtHPYYO+wAHLl9QTLUrmROlCpiNmwawNHtq4lS2sXo12J9FKJoBIgtzt6LOcl7puEDWrekOTnJvNUkPOCWTe2hoNt7DMrKnE4l0sKZBWp1nTai6TPvE/l/L9LuWjDLbbajFZ6QbTz+FhHqPBng5d1kZMsGTV6hxiQq2BEB6eZemttVcNR3cyrUZuw+oDOAYeSemLhG4w5ZVpNptyChMgFzC3yVDaa3zqkOAzZ4RRYj1gtJpqLp0lHPGXubSITq7q0az5zbeIfgpzSZks2HivGv1a4prPOoVmq1Ir1w/M2+owODK92j/9TqI2zms1Pgj0o36SAmbjBtFeMyJmcUNPb9VXdU4yThXBfVpVHWPsWeLArQZED11jjABYRTxOl378C7+yRw4=')).decode())
BASE_BUILD='7d8f96a4f42a85fc4c5fd79c8470dfb31ec444c357229200cb4b691c84fa8142'
MANAGER_BUILD='1bac4957fa2372eb3f2e57454db58cc93731597366781e1a07cb51128d97d6a4'
MANAGER_TEXT=zlib.decompress(base64.b64decode('eNq9fWtz2si26Hf/Cm3vuldoIjB+JRMonSonJjM+27F9bGf2OYdNqQQIrAlIbEk4Jg5V90fcX3h/yV2P7la3JDDJzD6pGQyt7tWv9V6rW/v7+zdBlIZjK0+DKI7iaTML4VcaLmbRKMijJG5ZZ7OZlUXz5Yx+W0E8thYJPF5ZX5L0c5hmUH0Ora1JmnwN49b+/v5eNF8kaW4FWe5awyALX5+41kOQPcyioWv9niWxa82D/MG1ksy1sodlHs3g7wp+5OF8MYlmIXyL5viZBqNwGIw+u9ZyGY1d62tEz/egt7m1ACAA0xL93cBPfjAO8hAByCfyN4P9msShHGO8nC9WMFIrXuzt/da7vbu4vrI8y46TPBwmyef2m+b4qN1u5kE0a47Dx3CWLOZhnJ+2aa2yk6PmyXHz5KT5eGjv3fY+9G57V+97/sU5AlnQ6vrTxdJHID4COTmFL4+H/uj49eHbyeHbn+GL3vLXs7tfsa3+/LQ9PHnz8/hoGAZvjt+eHIYn49Hw9XBydBwcTt5MXr89PHobjPDZ8GTSDl+/eRPCaG7Obi/u/8v/cNG7PL8DkP09C/7ZsBawJbkfjW2Xf/kT2PowXaRRnGNZnKTzYBZ9pR33ceOwNFngzwy/ZskyHYX+Ik3yZJTMbJch4/T88AnXVXsG1UdhHPpP85mfPQRHp6+xLF3GuBX4dZoGi4doRJCDEfU5TJbxOJNws2AOCBlPsQKgaR4FM/8hyvIkXWmjmUWTcLQazQhmmsxmyTL3g9E/l1EWIVAFjUYjGiVpNI1irWeY9zTGDYbqg729cTixRkmMaJhnfhzMw2wBGNmYJ+PlLMycDsGMM1je5zV9D5/CUWOUzBeApbJa31YwWouVPYDuVEEzDSdhGsYjGje2th0XIDoELQ3zZRrDTx7KY5hGk5WvmvjYEWzTMJpF+aoxWqZQDESnKogBzqMMCG70EGZe/3O4siZJauFfIFsTS6IJlv/FU7tNBD8K4iQGjjCTPSCQgfMXr3igeuRHA+r2EVYTSc8fLsfTMG9c39wDed3x1JLZ2CtaqQ4HrVGyWDWcbhx+8WR3lacEQZtF3w7jxyhNaOt8okzbtZlRiZ8DXgrRdWuRLBrQmPpRP6gGrEExL6hqzBNqO51iOVvBYhHG4wbQTMzMM4RJL5lXykEz2CDLQmQ4Sa5th2v3nhawxGbD4SwZfQ7HHSsbRTCfaBKNDgTBqLY4a/sV4kzr9ySKGwVMA3Pu02W4J5BH7kaBPpJKG2mSANqYWK2qeciyW7MkGGcNqnlgy4YtfGQ7rTQMxn4ePuUNx5htDYI4nmdyO9e+ieIYVqCYbdG3JXuyRg8BcKlxy9bhawjEFA377Hl3159uAfj7s/uzy+tf+vbJkT0gRNaqD5fRbCwZErR5d3bX8999urg83wAfaC9DNgJ1t3GEvi2kiD3Q90EB2iszFcBiwtAGfoilF70TQiF2nxy5J8fuyQmD5JnyiD9en3+67N2VOYxej/+0UKrjQPfrpRzLqiZQ0TgKCpmH4gpl276rqOcHAIDgPCUw/4jvb88uri6ufvHvej0UkvuvsjzluTPm5GkYeqA6tBZBmoUNHr3TBQEA3EvMJQN5kFMJkG2IFJh5iOkMIU7GoRcjMsbEI2JcQwTbGibjFdI3iIQ4ywPYjkbsYl9nWQZ83yEkCeJVQ6uQU4Ur2Gh+nLeisefZjGN3NvWQYw9xKw9SYHOZmAiNr4+DaeHXOGkedugXDNfnkoHXl4BwNV/BHqWNEvaq5Rk4r2D9zL21BfVTXwbliZUagSTNG/ujyRQoBKAg+p4c7QMRHm7DkXJ91ywxNnF/y6pX9nHzFnxYxiSCz8MJL3TcQuqCpS6JEVtb3hcRorvbLvT+8+bs6rx3Dhzg/JfevX/f+897bbUFEZdWW6NkUDfCtCBlQcabCZXqv0ipiud0BMm5JtvqEL/y7349c3W1rnMOqHMHs7g418n2e6BJg4CmY3eM/XbtQlT5MJPQ7tiGZmxvHI2zbbaNrfqy/erm9vr++v31JQmNfufwaODo09vSnNgPT4W4jTEdoKrN8LcN2D5qA4eehaM8HGM3VriIMsCszGocti0cTObAUhy2N1Y7VdW29gPmirLQSCZ0LdlBFzqoeQyAr3p/t7RdkdUFrUKtWejty8VbgHgCxfchB/1w8I94v1vHRLjRC5yDK2nbIr+8oLrKxXetQjmQZQ6MqAowyJN5NPJRAWnc9e4QoQ/sDeBZTXGtZxtYTr7MAGNvzu7uYPGquhAbPJ2yjiIQHfQ/Y1lJt7SSeLbqAgeUeyvsTnuNQ38lFq525eU/adcUStcV7OufsRU7dFUsgnyGlnXj92RY1avtAc5p82TEPgtaTEPgAgjITZMv+H/mdPFT6s3w3fnTJ1lBNR7F9af7m0/3rhjMTjuTLefzII2+hrKxQlOchDsLY5xb5rhgQQcz70Mwy0LnX0M/wjLBLmkZ/9dh2/PaFghaVeR5akCdWpL5o9PaDDEUVgxS0RSWOzN5bNGB6JDGq/BIiFdeCiFWN+DkZiVZGJd1liAL2QWI7jxMwfpAafTu7OpvBw2lXL2yhSkjTUC000SLVgRaOprzTkeMFGlTjHyUpGPdQJJtNllE3KKOrjyPpsJ2ClcKprCYU1R9TO4EdQ1pxeYwDNHLggnwvgcQ641inq6El4boxnsMaS2Bko11fAh4lvgBuCTbSBMJzDR2FhI1WcMg/mx9CTI0GsFkU4YZqgagBy5oOQiWG8xmyRd/EY0+wwAZmfSOqUULulmAstc4PnKF+reAdZ/g5ocNquK0AFDDxBjYPhru3gb/CFM+W7dI+BZSvnDaoHJZ1lfhMax6CkYomlJgNQImoFuxETtSG5XoiJIkbMWgwzm1y0+dYpvtq1zqsI8tGF9FR9+1lqLN9tUUlcz1/EFZIMmlMNijjGgHSaTwuED3eZjloKG0eKQ+9D2aJZkajSunqSDVzdVN82Tmtd0A/xyGzbcFpyQ2lKb+PJt69i3hKCDmBD3UwShNMsJUsCTyKMwODH0p69Y4X1piW8u+FJ2Uce+EIzFdxqjsbnA+AOVNh/RHVHMt+xp0Bit/CAFx0ozZ2MmRdfvLOwtVl1lIgxkFsQXrFgxnUfZgBRaqHwuNDAXRTcM4TGn4Onezi+LMZl63aRtBB7ZfoYe9hR8nDaf1ED4BTyzQu4CFBh00bc0/j6O0ET5FWe4nn4W5xd588tIpNHeLtvW0oD3fShG6ylfm4ptnJhi7+6z2s1qvs6G5u5EFdwwGXKBhicd2cGza2qmneaJNwXFcyWc7dauigWePNVDFLGHzitgTMhvmUK6sIVRR/3O4EtPDb4MaUOa8aP1L4kYBlftid7CvtVPrK9uNc+iMYoPHUsABRcUPY+g+1MU/hp/I9WZ5zKJl7AgRE7830KqJnryJHQ/bbziCg+2f8WPdtF1AX88+QHsc/f1CTGegvCDleZY0KiZ20crW65QpQGPHwhsIUMbRKG/oHgCjQtlvBw02eQZrmhVeBNmu7IcwG4GSCVYJMmHRRhWUm5BjAIEq/630bxYBD5pZkOXorHM5zOeJaJ8I9nky6IcUSEqSi3E/j4J/SeYlmcv8whNBQIWb2SrzMCAo99RTgUF0gnsUy1OxQa+IEiL38ihSKAKFnvhbgEZc8fDDlVFBT4UHZXTQk19coVV64q8IP6A++++99/f+7fX1vaf/KPr5+/Xt3+5uzt73PIWprkApTyCQqzwuHi140Xi7textf7wFDAkNr77YrSfgDRpB0UnVYPeqRa5pT5dDAEYUyjN+uRXbwquUFPEg3Gtig6h6o0Yn8LYFis88k445MgdwbmprDmTDLpa3vqRQn9V3/ChUgbJ8o6UQ26lgFNXjrC8L+53m8aC1BLs3baC8vTm7/xX0BQSwS3X0SEJ1HI0gwb794fb6v3tXEhKW/Na7vYBlO/fVl5uzi1v8fSKqqcaXF+9uz27/y0c87d3WAqmtsifos0ULFcXI+xttF+WcWk2n2BCcC0foFGOzXQ4VU+xZlurxOYQu9w1Dc7ShKCfqwc7nqJSs0O2Ibj8JkYtM/lr0URunJVXDnMoBbagrI7MyMDtdBmD3bXZ3K56rRaB9akTjcF4MRaiB7h6SuL2+vAT72j97/x+fLu4ukGFtCU8YC4DwPtKQGzgwr09DHbj5agFkP42TFEPHA1gHbT5NWFscUW5XloeBbF8ZoTmXJNmGlWmUQgS0Au9noHpgvEDEm7dOCT9q5sOjaIK6QdwNZtREb151SibWGaLULYvjPw3NdI1XchnoDNS6aIyxUuXafC68+4KEOQZZceXjp6YDmhEAlkI2SnG/FBtQ4VENIvAmWJU5giXdR4OLUwaCjYHaO884t47QBVoMkdhqC6QKzKXhkLkxjqZg6jQctdKbePh6XZscESzHUe6rwbH/QrF82KFJNMVNV7tZDa9qBmtWG+V9/uknXZcrsZfOtpAsojT2iPE0K4TV2qjprQvix2EKs1XkOaAhiREYWlptwKyXcpjB2xhC7EtgA6Mhmg+GGnNQWGAHo4dw9JncWmBFyvYHsq/CpjUhVhxjInMAa/NebMwdkP+4GomkIncBmjNse9DgL64ck6Mnhyj2WU5EqA7JyHeqGxVYpTiIq+vbj6SulObKMYWaufKDF+cpzWGqXawn+gPLa0zsXlYUsQyoaL+//nhz2bvvndvbII+XaTCchX6WhwsK7GI4yoSYLGB5g4zzHGwOuPrh00OwhEZjE7qRf0PcxbXlXsDXxyhbgrJorC5grNh8kXKjiXVz21WIgH8alib6rQRyeKJ2vz2g8fBPHJJ44LiWfRc9FREyQJpoAaYZO4esYbhKcAVg+Aq5MR4GquUIGS5weFvnNtUQUhY9CYzOfE7BwYgpWLHAHRTp+LyXxKCByQNhWQvOevlwc3xk9T6e4WZQDGm8An4TjWBkWX4AEi63irZrGXOeYYkvNrTB1rFM3CHfQOaRxdp5thEX7Q6us6vsevkb/fB2pz9Yb2CIa4NZgljCx6K3qkoNHM5mB79Npj1qHRmGKGxpBmjliAggQIVLjTM/Vp0XwoRIhgrWwK2JFWqSdxQucuuMUAbWrpemSdqpIw8c0gxs+AaBFxMYHNgyUAu0O50lw4b9kyBlp1sZOy6klC1yyX5PhjxVoA7ECpBhwLRkEpw5FBmc2DCCg0bhwTFDFdra6yELckrg5lRGKka7U8xCs0sKFiwdS2ipAPcxFs3VN0jwShVx29shSGcAq21YDh8WBh2hfR8RdeBJtDfgaehfQiRBCPi51qmdYQp3VOEKRBYA6xuNYO2ohtjNaZosF4gHXXbS4rfJLMjLWodLrRA5MtAiQe4zFElRGm4QXtHTPo/RRDEoEmGLzMQn7FWu0nOtDkiq5pLsEpEF27Gven/3z3u/9S6vbz72ru5P2/7NxeX1vW162ev9kGqQsOwD0xVZ8jAWsli0UdsycKo9/fTT82dySn5mDv+Z5Q2SgiFuKi5GI//TdkueepM5i7zkSuecMQziEo1WsML8bDlCsYDgQCLCqoZjrewLSCboDmiO8k/Jsy8fC6m7/uPaXT/lvZc7T/GXfrEWILnr1byYAsQSjklWjLovoIzqoiO/ubaIVsA6KG7ViWHtZwGKN63wtF2DRvqadrLlvAET0csGxVTVuOvWXratPqmFUD+QFGh751EcxMRzWZUnaVoFWozGAL3bICsdlDAnWeaw9sIefQk3S7jE7Gn7bv/0k0jioPSFjHIMXMRQHzUThabilxiNNF4egzQKhLdSY36VMdO6uOWF0kb7fTQikdHrT6klaYKM2riY03oqIR142q9DZah12japiILRwOSnfTGbQdGVhFBuoRajWPI6WhIA7Y74UlCXb+wQ6mwwXnPX6oZRg5NxBRQyBZ4UkNY8DGKBrRPQDkAiL1pYJms4uI78fTvq46ENlJRjn8OvlW4VeKgjoLvjcTLxDrkTbVj/dri1r3kkhwzfZJvacdrz4ElWDZ62VRV4zFYCKii1rgNDfGtiW/HqZwPlOpuwsrSTWirIFrRc64o3Kk+KdXio/+hksjbJRJvVS0oC0zis6zLzkfYrLE3ZqJIi+xVqbmqGLMfByzVMjw4vjcvsx0VVxtVIyNVGX1bMhst4jF552g83ikezJWiqj9E4TOT2sBfUY6XwoGGrYIKW1udPliDIGVjra7SwyR9pQGNMqW8dYhUYHjUVWe0hZgQG6crj/ltfovyBHEsNUYDfQbXP54tmNRTvdOdBHE2A00osRGNJ7SkCk0fjWv8dLT6gt1F1CXqJ7aqnFzeg4n24PLvvnTt48O1rSXFErd57prrkt8NvKrRCiVa0cqirphj+JOqh6ipNihCXirLlZALGN/Ft4am0W6PsEf/MUQlrLfBcF/wZT+zBuuJekH0WofJWKmwxu2RBCTr4rpHE5KylKfdlLpERoue5Os7Ao1OHel9/tS4YIaxsDmqfJc4qNWUAmdXNA5Vae4CuzoNZMrUkekCVEkRMB1ku0ChD6pdYTVkgEWAZEip0MVtZ2XJBxxejPLOwStbauHQi5vinrpu5g6h9gyU51bbzaT5Tu0qwBGJL879Ar1qVm3fEFkup6OmA07cruySmWLtNVZPLtLVqF8OwZTBRgbKZak1p5T412lSqAJdNk0cUCBRD0Gw3w7gzogzlTJj1RqjfgGD3pXA4eKaG5CtY05btb+xvK8jSKS/Q6cnY4+zRVX0hoYT0EMhCRhfi6kZZtXd9+nIkm1FkS7pdzaYi6plYvnlHdaIwKGC+OGHG8VUgNiO0Lhm2QxU+Jx2yoAmADvZoTnRDnyg9BlvoQ09MOSC8WtsbKIDMcckRhNJCQGpcDvLfV46CcwoXrawSQoXSwElObO0PVznaeTy1HHQ7p4Xmb/Q1rMluovzHtblZokekcZuFr696ZGwjp9V4OV9k0Dc7oDqyCnqccf19sR2gw5gbU6M5fgnp1IPWhANZNp8vRieFUrq0Sm2Y0ChZ4IGA43b1rEUOusoscwXGxcGsiZPiXEN77UbExb0jzcmWZCoDvBDbrBjI7FIAM1t11NPWMp5F8Wc894lOapWBp+tRDGAP/+3v76uTpip2ebDAmGGWkzCiOADQk5Z+SJFQ630yC4agMsKMwaIf5XTEv+7AlGfbtnHQVBz7FedgzbRtUdgnjVyET332tIByeywCE6pW4ax6gG35yvGJn2sBSn/LQ4LxSwnxsASQUAIep/AXnx/Vw6LD4uNwnlA8BL/4h22ZBQbb6RUjrKadd9kBVdQxDySXs9gzzyM3dIoHXRtHh+22e3R42nYc7dYFo8oxVjmmKq5lfwJ9lPNJn8KxOAEkckUrJ1bMU7TsFcopRpg54hwBFnCfWHIqAkWqFnBRQAk8DqeqCdQDkCjoKPGahgmjbLuHbRzmt6L0CEuP2odm6TGWHnPpswxDQLWj1+23R2/W5ZPU2rC/6QP+33IYrv2romSXqUhL9nWLsBDNDk8RiiM9Qcpnj8luMZerwA5xDwJ1SgiC+DTKbRPV2BROqQL159Nc/ae2XUl8BBIqndZWFkYtNY3yJ+953TXSDOpo01WWSlOcolS5BgDC4Brwu185czlQ/e7tlZ55m4YqD3JUcwy3HI80cxS3HYu7D2NLxNv4dBtM6UM0yVeE+fJJ5Qzc9rN2MTKKMBg9ALAYsZ6+OzuOo24EOzfeMokdBn5eOZSGBNaEj7fWl5CQGdToZJkBR19m7L0ohJxQirvAQEDbFZm5nG7O6fCCdFCOtTQ/uV0WfXw4hAaADKx5dHjytgt9R0INz4pRIYWFTyxSuwI+VUF+0Txsn7yVV8todLh981jeYqo8MkLB/PKHNAybxhU3yAlgjZvimhDZeb4cr/TJleAZK0GVLVKrh8ssj0HiaIxFjPOv1j2u50TkLVnhGA02heI4MWDaKGVBWYBVTKkfcd0OuktTYPsgwnD5WnVHpcQB5DFlLlFmZnHszh+CSgwMKViY/inlI8UwOnJ6YRBNBSt1n9cO/WAbwe0PUADgMdb6GLFxZEoC79SEy3tX5xdXv9iF7pSG/1zCeDMM6KPEp5hghHdhSD+XDMZly1nuabCKfAdQO+U0/XEaYBSPJIutYnR4VoeNDuHZevvzyfEbbZvl/TL+Mo5yGpbE4S5oRGCMQAGxcDyxgBglzpMEdDXSEwtcliTwdw4Dn2U6GsnpwoRGabRAq1XfNEEzgCMgXABGvupa4yTMdNPCKqEvddJUeQGuRYoK4ku2igGHYNpNxhEdrj6oij9Z7ScwPoFA4tFpW3mmMZtrLTYlnuLRJBBl42TeAkQMYI98KG3Q+gqsAHHw5EFhCyPMU1A3G20X4KFR4LEKgGrB3h+NacBq5EElAL8twYuyFFfec8OInLlp7VmFjhZx08PM9WY0D0baSH3UkIYr5R91Af5AhQ6cpnxM/lrzGXUKRS8dbxwUhsRwJXK8YWuCLEjTYNXg8bhjTH70yOvusEs/eIoyr60lNxOWZ54E0qftG2iVD/VDUUiU/QIzioEDpVb8xpW4guyEoTulmJxOLBu4mv/2dEEnURatfy4DMLlB9REzcPut9tGp23r75nTgtPKEdGZnbV7IgsNnDlo9vcoeI+H+cMm3QEdWO4qtw1g4+ErC9XdATouzcVZKdRvDbhBD17JDrRvRhSJxATENMfviACPkxFhYZLLh2rI+kMxZLIfyYjZuHih+a00A6APnQLKkuLm9/uW2d3fHIfy7Ul5EcWpIz4pQMxb5EGxr4MR57WgA2YOvXJON57Vr3366uiLGLvm3n8XBInvgnEJ9EO48+Cyd9Jk8L0NbUIUssozcwvJj7u+GmMhDx+NrOqRioxeyio3EqE3ZUxQGKkOU3j/29z9rEZ5w3KnU1oTobnlXztbEq90DId625JS9IviPMAp2S/CLYFTBHem3HqbRyfFLNMtgO5j8Bh7/bFSaKDbtikd1cSxnb2t6g2C2AkApSqbYb00ChGhRFPG56trxDcrTnoOeBtxKVJgslQ+IUIZxET1tRXwU6zf6CJ79DQdH7WpEf8BxTfmziDWW+y96Ic1h4NnFUIrrQ/jSCaEggNLcxL4t9Hq4VjJHRRNPMQFPzqSZiWpnvRZF6lvhdinTnJFhyuOukKV2urJyPUxdlMx27a0nKu1/LsNlSKlFIuLEXsblgnirD7LG7sBHQz8dKbNO8iTXPHd25xg1QvkQrJryo/oklgLui+7AXXLigaWTFgpaAaUd4cIUwty6BJPEeiCHJBprY+T0FCkAixRPWtB6P4ZH7V+/WiIlqYDNBGx3BMewVerGfAEMI8MTwB3BRkwy8jWGYnd07qINnIGZ7TTmY3e0H26RtdghjqXBoVCBSvlDZVKlpneebS2XrJRHtjmBbEvymD2GUY/olkg6hauAOvX7TaaQZCnOWoscVaNGFCTSA0TyJOtamy5Lh5FOyh37A8vp2awpIzsstO0dpI+tlAe65RSVjqaymYQ005QN0k+kZY0Rbe7JQrWUrtPUSBZlqt2hP649i4B9iHPjnb65qOf6TU9dS0dg0+rXr4qVFhZXQbdl8xiqtGBuuWlWF7yjVUrHs+8AFU7bbC1l0rgXphjZ+OYtSFkLuRm74snXAFuEJVogtWSzVXsE445zte8+nl1eGqnaog2w2li4XC268kRYYtMQFOE85dQiZZNZlGcLi0LaYY905EqvF7A7oPctYTX+TqKVWz3iPJTZaOFdvKJfGgkfBENvQyruPkBf5JBYdJfQU60OZ1Tp9qNuJFbGc29cDqxxCPQjsRhSauhpu8ndMx/qio1hhxSG80CftQi90KuFqfhZ7VaznQ1iA9AnEkNvktfUemojoN/FwCMEK3y9XZgG3RyLd0lKjKPjc0UG2RdMuk8lPhR3K6BvqTQIJfGwEyXuxFmJLrPrCHXwMs/mwI8izCnoTQvsM8+D0QNSSqWrXzmmIfxdGaIpcXjr4PBUuoNwUpxZ6nK+B7mKgCak5FeajSUFTKWfDyUaK/vbaLGAT2iOQd25p1QP4ZdbxnmyHNHaCYVVwyNvd7+Upp7Qqan6ZrojY+Bpvyon5URuT31yjhm+5l6djSB2GY0ApZUwPD5EPMoeFTRTghrhcSaY3Rrq+YTUWCvYBOElGc6AtIJNgGqD+5pZEUSzJZ2u1NR4fCw9hNt08Gpvy7ioLNFPdCk60m+D7Nt/RZEiriS1/t//+b/W+REQ0/3ZxaWy8q3yZYUalUz274i8O9azxEd5ymrdsn6TJI1eb0UMz/H64BjP7mhUw/e/ZW2UcIrxtrTrCkGUkzLY0ciWOMoWfU+6AV50khtTsr9Z97pctL5ZHyU3/Ga9l4oQfD+LVzyGb9adOahvAPJbs9mU/3e0D7vIitTyGju0IdIHNtn/Zj1X0xzXFhcXVposqTMV1wcg/fmxkZst29RgFTza1/DjFSAIzOSvmLqFcjgNWcthiYRORnvwqm83LfvVE02Isj4kIuhakcDXElRMuwNdJBE5XLQPqHlu5UflVEPH3L1PZGu9kGTIp7wfRCoayBr5kCP4Arp1HUvyBC6fJYAOpDNgbAhVjLG6/ETBFpu7G0edj21HvzkB70/VL0ylC1VVtAB5gnZ0jK/jnwc5LF0+i4bqSVHUAn2jYZ9Np9rRp0q71mKF3zCzcTHLC8JGhVmrBQZDkmay+SUmQYzfY9k8WLjvUImAGV3BzwJCNHWDJ2AyALaVLYcIKWscuscunvwjf/bha/fE+dHjHaOHBPbjz0tcH5SPh3EHHZxDvz1o4fgb27PIqcHA7U+rCbvVWnbStN1ZMAxnanjaJQaiU4zRP4Ei/Bl4dRFwWAFpeY1mq33qHsKH4z4xHNtgXDbU4+ISd5qkfHm/7eZRPgs9IxaKw82sxnIBdCnCuY7tdOWAZuEUGZSjp1JrQpBX63DQGgZpg1JEx5UFYxce+e+0lrBsKid6c/r0oL5taeGg/+DpAYmo0XYJcz17OAtGn2HBofBLNM4fdNe8bISrLdb20D18cVnZHrFosHQRWWVhpbRgU0Aml2iOAEcn2zR6wjAEMrhG4zUGelwYB4chgN84dSfZTK5AG+Id/VTgSotCEg3KcCzvwysu1glF0prT/Z0vFqsGd5posRo98tD7kfv7AMfJUGukS2nBj2BwczDavzQYgBtkCxDInh0s88R2R8BZPJPRNEBfGdM/lB+jt6dvT0b47XT09uc3R+iowMPTns6OGv3mYevUbcL/LaSW04F77FQHQjvPdMZ5Na8dl37TVgPxTWzOG7Se0RRc2xvc0JKHhXX8q3S6RNI15vq4R5jF457Af28H7pPes8hrwjqHR/hxjB8n9IF1BTIal00yngok/CUNVtKH0bWwEgq1LnDLMIwLdUrDRTxvnqON5s+CVbLMG04Xi7LgMYS/SrjpMo3DQxnnM48XkXf4uv0drcYTQDkUFHzHoTrADtxFD04UmL7pCAPHSLqbHldS/lhZMW7DLPL5ZLcNYfV0zOvbTH3X9PnyraTSQqs7jyJ9x7IWu5RU9nRdV1u9qgO83hYB0v22ooVwcw4cz3tdvhFgy0mZQlpVAHE+SYAZa/oFDrubkJtffsFQPe5S5Fc+4iGYHS5zFSkP2pGj4u6Zl8+cuLupmoPSbVhi0nQNlpgDX/JkHAugIjO/99/a24+iYJO6wyaik690OSg0ajgGuhQMWZx90XboKy37hjzh+gs2ZJ1+Ne2X0JRojaZY1HwRRes6Qox9BjuXb9ktzOACrDg7vxa5lpUHovwr+b85eu00Dytp3UJoVppXk7n5Wl6xaHzFI18B3NXGzNfsiis4OZub42alu3vEdbzanT2ijbwr2Dy3LGOzNbu3A3nV35UiYQrqKmUlFPnpuH4yP72Sg05oqfLWa/DaPJZWvfpD+esFY7U73H99gAE45FpeVZQnfpFR7UcTyaVDkzcLn8nZp/tr//zi7v311RVe0XN95UveXJNoBRXP3l0C114Xt/MiTE7qEtWcvxj8HZFJq1QzfGgAE+iUL8XV+r3728XNTe/cv7q+9wvQPAg9FuK9KJiqJ9zIfpsmyXSG16ljQrow3cR7kdxxGj0WXEPd+jObUYh/ipY+aO+ytr2MA7pnzeYLaAALOM1dVGjJ58gX4PsjaBgIqGW/2AUNxLUnsyWmM8RjfxnP8QB1uadzrGdRtfpOxAUmPfpD795DN/RI0xlq/JUaUum3D7naNn26Ovvt7OISUQQFBUd+0LQB4EgNzFD1DbM7+i/t0Ab5DRp42zKqPSzeWtYZYDeowNFIOzSgz6+rlR+MQ0pfQS+EsQPI1JZ45KGF3kVYJO14Qz366fNai6yaW7ohigIoYAsOQ2CamBFHG0K+Od4FPnaCRDBZpjAUESCxkoW8PxnYLd5G3dr7Awt/2/uPT707oAn/7MN977a4o/Hu7DfcitqI9ovbYaZT4jkYvggmTZIJLGuQs2MoTB+jkTrcQWabHDBdLMQrKzb0jPMxlYsTZcAwAcEu6FQFN1rWB9wbtCVpJV3sDLUsGgldtyS2Uzs/Yn0i91hHc3N9l4esgg8GpyD6a1XITzPxyzTe2IZVatPs9W5UKZewjgjG0djMkqUAhKA8M1QjaYrSy1QsSWAw7qhY45Z1XktMP0xGYsL+B6Alyq8tcYi1TOsfJTAsv5AUdJHyU67OVMqkW1213EHg63JPB7HlLQyimq5gGC0rCjpm4XCFwt4x5GGlC6l3yFbbbSBVbTf1sQx8u31g9lBOtiGnR126jXnaKG8UvZWzMgZ0f8DJEez9yTF+nEjc50Q573m90x2KANLb0ot2LeGecZJWO9Vp3kkozOsDMqrH/ixJFmZGEYDvF2kkA11Q8Yp+56sMu6zfau9/lLWdsh4gano8BHMD5bsHN6GN7oBS7xZIcvnGiZ3fayXeZMKvs+IB0XusysDl8La9NLHYHDb6VBP1Kk5PXoq9Gfzmay2KOnUIbEboCp8i3wPqvXAL56Z7NcvXtHZqr0wv8WHjPFKn7jDTevP8yy+Z3HCd+g7vx1KZuduuvNtwaWmlrzhrcRad0F48SnBWt3ervgy24zFCaRfsMTcQicVxJlSuaoITJTUfsCePMpBcC12I1hdUFeKEThmxvhXF+ltEAC5oGLtk7pa4aMNM5SJGpSdx8V0vDTzWcixIT/B7mXQs2JYQdcvYr2OpUtDhxQrBrOYedLf0eiVObNTey1NKjy4x2W7pMR30U8nPXZGaqdKvsQAPdtPl9S3MQ+Sbm/AC8aqatPF8ofGS3g2Xw77EiA/0N2Fvuyj+xXfSVt+cQFuNWORtuvC3W/tqOLymk7DM5yR5Vs/pu6Nrdu+CLNzN5lKgRVC1qvErHQpKpqyiV143B3ykPIVy2k7FRlMvPLA76msL004DvKwVDQh8UR3mHKaGB6K91gM0yg8tZ+KqGdWraaXrOFgZ1956oam/d4iLy4V8m05XhMRB4f2RPmo11yDKwj1DuD/XvOIS9F9Y+rR0nfXHs6uzX8AEEzdab7/w+gcym6WQ7AhqcpWk7ZhydscTWVtSl19KTdbfTLPruw4pwwnVDkwT5ZceRvP5EnPlMMWPXn1FbwDrIqN/LXIh+bgqkAi+mMk2OqZ3wuMBx5U2/MMdTuntlFhNqaOc0SSyYPie3SJHRmWqdYv81Dj5Qnk2yzmGkDAMq23hWp6h0QTgJi1QCNHvZJENWylzvqHv8P6fHPknx/7JycZ3k2pdixfR8HuZ+MSLW3ovUyF0vFJGmPE+s8zWata+4UnaT0ViWUl/VjfPbr15e1Pr0rXb1cAIJkJ36hwwpctqzC2qa/CiQKjl9cW5h0VhzCjOX3bsJrEykO3OFi+u6X7p1WSq4dHIxTIHBiqUCvsfsbx7RCXGiRfKyrhpZj1hHP9ABPafSgnSJ0cHJ8cHJydmktoL7hVh94Oasc0DoKdTqBb1r4bTdB6RjempFkYldT3097mRS/dBN67v6CZo92/hir/8htogfzUvi1Z3pr0LxiKSVXNNTq3LzFLXNinfDYaqScTg+o+DOUil8YG4vaW4EkzMAH14FotvTmHf4L4pdoUWbvMpOs234XT/wCq+QBNmcu2GA9tIdFCkgifqjkvmmb7Y/lXdtcO6APS5X3ECB81jZT9UpcUugkR7P9zYPGFkM/WxDiYuhfs+V/m/UOWqYmHh5RViqCBCumK+uJSubHdRHAgRD1lBjaP2z3HYlryPWgzvx5XwTVjkv6ydl/VrU/veUdmuw8r/MXVbpfE2efZyv1eFAk53F/xr9O/NXEcdDa47+FsN8u1wZl9M+B+x53lWraDMzew2vscUam9GwzjzNrxqsFvr5NCvxUfnAOjA9qBRRu2XOLJcG7Nd2ebf6vIovZUsnedpGNId+urtc8YRW+ExKL0tZCeak23P3l3fyhOj2huaoC5K1GESpGNKrk6Xi9wRx9YEoQnnBVT9MXNaHDT9AXN4bV6iZrxx4vt2TYyB+sZh0ElA+lYc5628mGFralWn/thJSRobQkaszIuSu/IyjMoWazcK+GHNCzLqtkd7I6rW+ru4bKXb79hEbV153eX1nIxdFdB1+/uSEKAhrcvdfVdSWBGtql5yCUv1J6WbdRmWdOdKIpWT4iUpplRVHH/w2BR36+z9QbWwTBilNyh0ajyZTeHlLLvIdtPYNiuB1f2voez/MXmuFK8/rmcJiOWzH3+qP06nRWXhkYJgFTRoKnv/HxtAeL0=')).decode()
assert hashlib.sha256(json.dumps(BASE_MODULES,sort_keys=True,separators=(',',':')).encode()).hexdigest()==BASE_BUILD
assert hashlib.sha256(json.dumps({'manager':MANAGER_TEXT,'base_modules':BASE_MODULES,'source_catalog':SOURCE_CATALOG},sort_keys=True,separators=(',',':')).encode()).hexdigest()==MANAGER_BUILD
for filename,text in BASE_MODULES.items():compile(text,filename,'exec')
exec(compile(BASE_MODULES['contracts.py'],'base-contract-helpers','exec'),globals())
exec(compile(BASE_MODULES['reporting.py'],'base-report-helpers','exec'),globals())
exec(compile(MANAGER_TEXT,'expanded-development-manager','exec'),globals())
validate_expanded_budget(OPTIONS)
atomic_json(SESSION/'release_identity.json',{'version':VERSION,'manager_build_sha256':MANAGER_BUILD,'base_build_sha256':BASE_BUILD,'sources':SOURCE_CATALOG,'training_performed':False})
(SESSION/'expanded_manager.py').write_text(MANAGER_TEXT)
print('Evaluator identity verified.300 NEW development episodes; no historical imports.')


## 3. Automatic execution and durable episode resume

300 new trials; no training. Progress prints every30 wall seconds and20 control steps.

In [ ]:
EXECUTION=run_expanded_development()
print('Execution status:',EXECUTION['status'])


## 4. Review and output files

The exact aggregate directory is printed:
`MyDrive/DiffusionPolicyEvents/results/closed_loop_development/d200_tail_development50_seeds42_43_44_v1_<hash>/`.
Upload `expanded_development_full_bundle.zip` (all verified evidence/videos).
The smaller `expanded_development_evidence.zip` excludes videos/trajectory arrays.
On failure also upload `expanded_development_session_evidence.zip` from the printed
`notebook07_development50_sessions/<UTC_session>/` directory.

Reports include per-model counts, conditional descriptive Wilson intervals, complete
training-seed variability, paired differences, success/failure heatmap and an unsustained
episode index. A descriptive bootstrap resamples reset IDs JOINTLY across the six fixed
models; it does not include training-seed/model-selection uncertainty. No automatic
winner, significance claim, retraining or robustness experiment follows.


In [ ]:
from IPython.display import display,Markdown,Image
if 'OUTPUT' in globals():
    if (OUTPUT/'expanded_development_summary.md').is_file():display(Markdown((OUTPUT/'expanded_development_summary.md').read_text()))
    if (OUTPUT/'development_results.png').is_file():display(Image(filename=str(OUTPUT/'development_results.png')))
    print('UPLOAD:',OUTPUT/'expanded_development_full_bundle.zip')
print('SESSION EVIDENCE:',SESSION/'expanded_development_session_evidence.zip')


## 5. Completion-only resource release

The flag is defined in cell1. Verify saved results before Drive flush and runtime disconnect. Partial/failure paths never call this automatically.

In [ ]:
DISCONNECT_STATUS=auto_disconnect_if_complete(EXECUTION)
print('Disconnect request status:',DISCONNECT_STATUS)
